# 08 · ReAct（含 Plan-and-Execute 脚手架）

- **ReAct：** Thought → Action → Observation；**无 tool call = 结束**；设 `recursion_limit`。
- **Plan-and-Execute：** 先 Planner 再 Executor；另限 **max replan**（本册后半只读结构）。

本册用确定性轨迹验收公开 Action / Observation / 停止条件。


In [ ]:
from __future__ import annotations

from pathlib import Path
import os

from dotenv import load_dotenv


def find_repo_root() -> Path:
    """向上找到含 pyproject.toml 的仓库根，避免 notebook 工作目录不在根上。"""
    here = Path.cwd()
    for p in [here, *here.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return here


ROOT = find_repo_root()
os.chdir(ROOT)
load_dotenv(ROOT / ".env")
print("cwd =", ROOT)
print(
    "model_configured =",
    bool(os.getenv("LLM_BASE_URL") and os.getenv("LLM_MODEL")),
)


In [ ]:
def show_graph(graph):
    """在 notebook 展示图结构 PNG（课表：看得见 State / Node / Edge）。"""
    from IPython.display import Image, display

    try:
        png = graph.get_graph().draw_mermaid_png()
        display(Image(png))
    except Exception as e:
        print("PNG 不可用，打印 mermaid：", e)
        print(graph.get_graph().draw_mermaid())


In [ ]:
from typing import Annotated, Literal, TypedDict

from langchain_core.messages import AIMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode


@tool
def search_rules(query: str) -> str:
    """检索订舱/危险品规则摘要。"""
    if '锂' in query:
        return '锂电池：MSDS + 危申；核对两港限制。'
    return '请补充港口与货种。'


class ReActState(TypedDict):
    messages: Annotated[list, add_messages]


def decide(state: ReActState) -> dict:
    """确定性 ReAct：首轮提 tool_call，收到 ToolMessage 后给出终答。"""
    last = state['messages'][-1]
    if isinstance(last, ToolMessage):
        return {'messages': [AIMessage(content=f'根据工具证据：{last.content}')]}
    return {'messages': [AIMessage(
        content='',
        tool_calls=[{
            'name': 'search_rules',
            'args': {'query': last.content},
            'id': 'call_search_rules',
            'type': 'tool_call',
        }],
    )]}


def route(state: ReActState) -> Literal['tools', '__end__']:
    return 'tools' if getattr(state['messages'][-1], 'tool_calls', None) else '__end__'


builder = StateGraph(ReActState)
builder.add_node('agent', decide)
builder.add_node('tools', ToolNode([search_rules]))
builder.add_edge(START, 'agent')
builder.add_conditional_edges('agent', route, {'tools': 'tools', '__end__': END})
builder.add_edge('tools', 'agent')
agent = builder.compile()

show_graph(agent)
print('mode = deterministic ReAct trace')


In [ ]:
result = agent.invoke(
    {'messages': [{'role': 'user', 'content': '上海→洛杉矶 锂电池订舱要什么文件？'}]},
    {'recursion_limit': 6},
)
for message in result['messages']:
    print(type(message).__name__, ':', getattr(message, 'content', '')[:200])
assert any(isinstance(message, ToolMessage) for message in result['messages']), '未出现 ToolMessage'
print('PASS: ReAct trace contains ToolMessage and final answer')

## Plan-and-Execute 脚手架（结构示意）

不必完整实现。记住：Planner 产出步骤列表 → Executor 逐步执行 → 可选 Replan（**max replan 1–2**）。

In [ ]:
print('''
Goal → Planner → [1],[2],[3] → Executor → Result
                 ↑________ Replan (capped) ________↓
探索用 ReAct · 长链用 Plan-Exec · 可组合
''')
print('08-react scaffold noted')